# Travel Log Generation – Full Test\n",
    "\n",
    "This notebook reads all corrected pages from `data/corrected_transcriptions/`  \n",
    "and condenses them with **Qwen3-4B** into a continuous, coherent travel log.\n",
    "\n",
    "**Pipeline:**\n",
    "1. Load all pages in sorted order (B1_P012 → B1_P074 …)\n",
    "2. Group pages into batches (~5 pages per batch)\n",
    "3. Qwen3-4B normalises dates and weather conditions, writes coherent entries in English\n",
    "4. Compile full document → `data/travel_log_full_test.md`\n",
    "\n",
    "**Voyage context:**  \n",
    "Johann Reinhold Forster's journal of Captain Cook's second voyage (HMS Resolution, 1772–1774).  \n",
    "Departed Plymouth: 13 July 1772. Undated entries in Book 1 → assumed 1772.

In [1]:
import gc
import json
import re
import warnings
from pathlib import Path

import torch
from tqdm import tqdm

warnings.filterwarnings('ignore')

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

CORRECTED_DIR = REPO_ROOT / 'data' / 'corrected_transcriptions'
OUTPUT_PATH   = REPO_ROOT / 'data' / 'travel_log_full_test.md'
JSONL_PATH    = REPO_ROOT / 'data' / 'travel_log_entries.jsonl'

QWEN_MODEL_ID = 'Qwen/Qwen3-4B'

BATCH_SIZE         = 5    # pages per batch
MAX_SUMMARY_TOKENS = 1024

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device        : {device}')
print(f'Corrected Dir : {CORRECTED_DIR}')
print(f'Output        : {OUTPUT_PATH}')
print(f'Model         : {QWEN_MODEL_ID}')
print(f'Batch size    : {BATCH_SIZE} pages')

Device        : cuda
Corrected Dir : /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/corrected_transcriptions
Output        : /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/travel_log_full_test.md
Model         : Qwen/Qwen3-4B
Batch size    : 5 pages


In [2]:
import importlib
import subprocess
import sys
from transformers import AutoModelForCausalLM, AutoTokenizer

if importlib.util.find_spec('ftfy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy'])
import ftfy

gc.collect()
torch.cuda.empty_cache()

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)

try:
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID, dtype=torch.float16, device_map='auto'
    )
    load_mode = 'float16'
except (torch.cuda.OutOfMemoryError, RuntimeError):
    print('float16 zu groß – versuche 4-bit...')
    if importlib.util.find_spec('bitsandbytes') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'bitsandbytes>=0.46.1'])
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4',
    )
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID, quantization_config=bnb, device_map='auto'
    )
    load_mode = '4-bit NF4'

qwen_model.eval()
print(f'Qwen geladen ({load_mode}): {QWEN_MODEL_ID}')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {used:.1f} / {total:.1f} GB')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen geladen (float16): Qwen/Qwen3-4B
VRAM: 7.5 / 11.6 GB


In [3]:
def sort_key(path: Path):
    m = re.match(r'B(\d+)_P(\d+)', path.stem)
    return (int(m.group(1)), int(m.group(2))) if m else (99, 9999)

all_files = sorted(CORRECTED_DIR.glob('*.txt'), key=sort_key)

pages = []
for f in all_files:
    text = ftfy.fix_text(f.read_text(encoding='utf-8').strip())
    pages.append({'page_id': f.stem, 'text': text})

print(f'{len(pages)} korrigierte Seiten geladen.')
for p in pages[:5]:
    preview = p['text'][:80].replace('\n', ' ')
    print(f"  {p['page_id']}: {preview}…")

36 korrigierte Seiten geladen.
  B1_P012: === Buch 1, Seite 012 ===   ms. germ. quart. 222   Journal of a journey   from L…
  B1_P014: === Buch 1, Seite 014 === Journal of all the incidents previous to my Appointmen…
  B1_P015: === Buch 1, Seite 015 === my name, qualification & place of side &   added he th…
  B1_P016: === Buch 1, Seite 016 === May 27 I heard that Mr. Divington's letter had been re…
  B1_P017: === Buch 1, Seite 017 === warning: for which purpose, still, not me the Question…


---\n## Prompts\n\n**System prompt:** historical context; Qwen acts as a maritime historian.  \n**User prompt:** batch of ~5 pages → Qwen returns:\n- Normalised dates (format: `DD Month YYYY`)\n- Normalised weather conditions (standardised English terms)\n- Coherent travel log entry in English

In [4]:
# ---------------------------------------------------------------------------
# Weather normalisation glossary
# ---------------------------------------------------------------------------
WEATHER_GLOSSARY = """\
Normalise all weather descriptions to these standard terms:
  light airs / light breeze  → light breeze
  gentle breeze              → gentle breeze
  fresh breeze / fresh wind  → fresh breeze
  strong gale / hard gale    → strong gale
  gale / storm               → gale
  squall                     → squall
  calm / dead calm           → calm
  cloudy / overcast          → overcast
  clear / fine               → clear
  rainy / rain               → rain
  foggy / thick fog          → fog
  hazy                       → hazy
  rough sea                  → rough sea
  smooth sea                 → smooth sea
  heavy swell                → heavy swell
  moderate sea               → moderate sea
"""

# ---------------------------------------------------------------------------
# System prompt
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """\
You are an expert maritime historian specialising in 18th-century exploration.
You are working with the manuscript journal of Johann Reinhold Forster,
naturalist aboard HMS Resolution on Captain Cook's second voyage (1772–1774).

Voyage context:
- Ships: HMS Resolution (Captain Cook) and HMS Adventure
- Departed: Portsmouth/Plymouth, 13 July 1772
- Book 1 covers: May 1772 (pre-departure preparations) through approximately August 1772
- Original language: 18th-century English with nautical, botanical, and zoological terminology

Date normalisation (always "DD Month YYYY"):
- Assume year 1772 unless the text indicates otherwise
- Examples: "July ye 16" → "16 July 1772" | "May 27" → "27 May 1772" | "July 4" → "4 July 1772"
- Mark uncertain day numbers with "(ca.)"

Weather normalisation (use only the standard terms below):
""" + WEATHER_GLOSSARY + """\

Your output: a coherent travel log section written in clear, fluent English.
"""

# ---------------------------------------------------------------------------
# User prompt template
# ---------------------------------------------------------------------------
USER_TEMPLATE = """\
The following {n} pages from Forster's corrected OCR journal should be condensed
into a single coherent travel log section.

Instructions:
1. Extract all date references and normalise them to: DD Month YYYY
2. Extract all weather mentions and normalise them to the standard terms
3. Write a continuous, readable travel log entry:
   - Maintain chronological order
   - Use normalised dates as structural headings: **DD Month YYYY**
   - Open each date section with a brief weather note: *Weather: ...*
   - Cover events, observations, locations, people, and natural history notes
   - Paraphrase freely – do not copy raw OCR text verbatim
   - Mark missing or ambiguous dates as (undated) or (ca.)

--- PAGE CONTENTS ---
{pages_text}
--- END ---

Write the travel log section now:
"""

print('Prompts defined.')

Prompts defined.


In [5]:
def run_qwen(messages: list, temperature: float = 0.3,
             max_new_tokens: int = MAX_SUMMARY_TOKENS,
             thinking: bool = False) -> str:
    """Führt Qwen3 aus. Thinking optional."""
    msgs = list(messages)
    if not thinking:
        last = msgs[-1]
        msgs[-1] = {**last, 'content': last['content'] + '\n/no_think'}

    prompt = qwen_tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(prompt, return_tensors='pt').to(qwen_model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.05),
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
    if temperature > 0.05:
        gen_kwargs['temperature'] = temperature

    with torch.no_grad():
        out_ids = qwen_model.generate(**inputs, **gen_kwargs)

    new_tokens = out_ids[0][inputs['input_ids'].shape[1]:]
    text = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    return text


def summarize_batch(batch: list) -> dict:
    """
    Fasst eine Gruppe von Seiten zu einem Reiseprotokoll-Abschnitt zusammen.
    Gibt ein Dict mit page_ids, raw_text, summary zurück.
    """
    pages_text = '\n\n'.join(
        f'[{p["page_id"]}]\n{p["text"]}' for p in batch
    )

    user_prompt = USER_TEMPLATE.format(
        n=len(batch),
        pages_text=pages_text,
    )

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_prompt},
    ]

    # Etwas mehr Tokens für längere Batches
    dynamic_max = MAX_SUMMARY_TOKENS + len(batch) * 150

    summary = run_qwen(
        messages,
        temperature=0.3,
        max_new_tokens=dynamic_max,
        thinking=False,
    )

    return {
        'page_ids':   [p['page_id'] for p in batch],
        'page_range': f"{batch[0]['page_id']} – {batch[-1]['page_id']}",
        'summary':    summary,
    }


print('Inference-Funktionen definiert.')

Inference-Funktionen definiert.


---
## Verarbeitung aller Seiten

Jeder Batch wird einzeln verarbeitet und direkt in die JSONL-Datei geschrieben  
(resumable: bereits vorhandene Batches werden übersprungen).

In [6]:
# Bereits verarbeitete Seiten aus JSONL laden (resumable)
done_page_ids = set()
existing_entries = []

if JSONL_PATH.exists():
    for line in JSONL_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip():
            entry = json.loads(line)
            existing_entries.append(entry)
            done_page_ids.update(entry['page_ids'])
    print(f'Bereits verarbeitet: {len(done_page_ids)} Seiten aus {len(existing_entries)} Batches')

# Noch ausstehende Seiten
remaining = [p for p in pages if p['page_id'] not in done_page_ids]
batches   = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

print(f'Seiten gesamt     : {len(pages)}')
print(f'Ausstehend        : {len(remaining)}')
print(f'Batches zu ver.   : {len(batches)}')

# JSONL im Append-Modus öffnen
all_entries = list(existing_entries)

with open(JSONL_PATH, 'a', encoding='utf-8') as jsonl_out:
    for batch in tqdm(batches, desc='Reiseprotokoll-Batches'):
        entry = summarize_batch(batch)
        all_entries.append(entry)

        # Sofort persistieren
        jsonl_out.write(json.dumps(entry, ensure_ascii=False) + '\n')
        jsonl_out.flush()

        tqdm.write(f'{entry["page_range"]}: {len(entry["summary"])} Zeichen')
        torch.cuda.empty_cache()

print(f'\nVerarbeitung abgeschlossen. {len(all_entries)} Einträge gesamt.')

Seiten gesamt     : 36
Ausstehend        : 36
Batches zu ver.   : 8


Reiseprotokoll-Batches:  12%|█▎        | 1/8 [00:24<02:54, 24.95s/it]

B1_P012 – B1_P017: 3054 Zeichen


Reiseprotokoll-Batches:  25%|██▌       | 2/8 [01:00<03:08, 31.46s/it]

B1_P020 – B1_P028: 4470 Zeichen


...

B1_P056 – B1_P064: 3527 Zeichen


Reiseprotokoll-Batches:  88%|████████▊ | 7/8 [03:28<00:29, 29.66s/it]

B1_P065 – B1_P073: 3667 Zeichen


Reiseprotokoll-Batches: 100%|██████████| 8/8 [03:38<00:00, 27.26s/it]

B1_P074 – B1_P074: 1289 Zeichen

Verarbeitung abgeschlossen. 8 Einträge gesamt.


In [7]:
# Reload all entries from JSONL in correct order
all_entries_sorted = []
for line in JSONL_PATH.read_text(encoding='utf-8').splitlines():
    if line.strip():
        all_entries_sorted.append(json.loads(line))

def batch_sort_key(entry):
    pid = entry['page_ids'][0]
    m   = re.match(r'B(\d+)_P(\d+)', pid)
    return (int(m.group(1)), int(m.group(2))) if m else (99, 9999)

all_entries_sorted.sort(key=batch_sort_key)

md_lines = [
    '# Travel Log – HMS Resolution (1772–1774)',
    '',
    '**Source:** Johann Reinhold Forster\'s manuscript journal,',
    'transcribed and post-corrected via TrOCR + Qwen3-4B.',
    '',
    '---',
    '',
]

for entry in all_entries_sorted:
    md_lines.append(f'<!-- Pages: {entry["page_range"]} -->')
    md_lines.append('')
    md_lines.append(entry['summary'].strip())
    md_lines.append('')
    md_lines.append('---')
    md_lines.append('')

OUTPUT_PATH.write_text('\n'.join(md_lines), encoding='utf-8')

total_chars = sum(len(e['summary']) for e in all_entries_sorted)
print(f'Travel log saved : {OUTPUT_PATH}')
print(f'Entries          : {len(all_entries_sorted)}')
print(f'Characters       : {total_chars:,}')
print(f'Words (approx.)  : {total_chars // 5:,}')

Travel log saved : /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/travel_log_full_test.md
Entries          : 8
Characters       : 26,143
Words (approx.)  : 5,228


In [8]:
# Vorschau der ersten beiden Einträge
from IPython.display import Markdown, display

preview_text = '\n\n---\n\n'.join(
    f'**[{e["page_range"]}]**\n\n{e["summary"]}'
    for e in all_entries_sorted[:2]
)

display(Markdown(preview_text))

**[B1_P012 – B1_P017]**

**12 May 1772**  
*Weather: clear*  
We departed from London on this day, with the intention of journeying to Plymouth. The weather was clear and favorable, which was a relief after the long and arduous journey from the city. I had not seen Mr. Irwin since my arrival in London, and he had approached me in a mysterious manner, informing me that Mr. Banks would not be joining the expedition. He asked if I would be willing to go, provided that proper provisions were made for my family. I expressed my willingness, but with the condition that my son, George, would also be included, as he was both a nationalist and well-qualified for the role of a draughtsman. Mr. Irwin emphasized the importance of keeping this matter secret, as he had the means to ensure my employment in the expedition. I was instructed to keep the details confidential and to maintain the strongest secrecy regarding this matter.

**4 May 1772**  
*Weather: clear*  
I met with Mr. Baines Barrington, who informed me that Mr. Banks had not gone, and that he was as keen as to know whether it was two of the expedition. I told him that I had already heard that Mr. Banks had secured his equipment and supplies to be taken out of the Resolution, which was currently repairing at Sheerness. I had learned this from an acquaintance who was well acquainted with Mr. Stephenson. Mr. Barrington then expressed his approval of my condition and desired that I carry a letter to Lord Butler on the subject to the Admiralty. I brought the letter to his house, but he was not at home.

**27 May 1772**  
*Weather: clear*  
I heard that Mr. Divington's letter had been read by Lord Butler at the board, and that Lord Palmerstone had added a very good character for that man, believing he could do in that branch as much as any man in the King's warrant. It was not certain whether Mr. Banks would go or not, but it would be decided by next Wednesday, the 11th of June, or then Mr. Banks would have a mixed audience with His Majesty. I had my day at the Club of Banks past men's, where it was made into my recovery. Lord Sandwich suspected that Mr. Banks would endeavor to obtain a new ship, and he went to His Majesty, saying the plan of sending me on the expedition with the character given me by Mr. Barrington before the ring, who was graciously pleased.

**32 June 1772**  
*Weather: clear*  
Lord Sandwich came to see Mr. Barrington, and he seemed pleased with the plan and my character. As he heard of my readiness, his Lordship was still more pleased. Dr. Solander told me that it was not certain whether Mr. Banks would go or not, but that it would be decided by next Wednesday, the 11th of June, or then Mr. Banks would have a mixed audience with His Majesty. I had my day at the Club of Banks past men's, where it was made into my recovery. Lord Sandwich suspected that Mr. Banks would endeavor to obtain a new ship, and he went to His Majesty, saying the plan of sending me on the expedition with the character given me by Mr. Barrington before the ring, who was graciously pleased.

---

**[B1_P020 – B1_P028]**

**16 June 1772**  
*Weather: Clear*  
On this day, the Chief Members of the Council of Rural Society signed a Certificate in my favour, recommending me as a proper person for going on the Expedition. I was pleased to receive this endorsement, as it was a significant step towards my appointment. Mr. Hudson spoke to me about the affair and told me that he had recommended me to Mr. Dyson and to Sir Gilbert Elliot. He also mentioned that there had been an intention to make a motion in the House of Commons on Friday last in my favour. However, Mr. Thrips and Mr. Burke had been opposing it, and it was expected that the motion would cease. There were great difficulties in regard to the matter, and Mr. Burke, Mr. Burke, and Mr. Philips wanted to have an interview with Mr. Hughes Palliser and Mr. Stephens at the Speakers, but Mr. Sandwick had already seen the streetess in of them, not to go. The House of Commons was unable to bring up the matter, and on account of the resolution of the house having not mentioned my name, the many could be applied to any other person giving on the same Expedition.

**4 June 1772**  
*Weather: Clear*  
I was informed that the House of Commons had not brought up the matter, and that the resolution of the house had not mentioned my name, so the many could be applied to any other person giving on the same Expedition. North spoke to Mr. Barrington, who argued the case and showed that as the resolution of the house had not mentioned my name, the many could be applied to any other person giving on the same Expedition. North having mentioned the case to His Majesty, the money was ordered to be issued to me from His Majesty's civil List. It was stated that at the meeting of the House of Commons at shows, it would be decided whether the money should go to me and Mr. Lind.

**4 June 1772**  
*Weather: Clear*  
From this day I could first consider myself as appointed by His Majesty on the Expedition. The Order signed by the King imported that 1795. I should be paid to me by the Exchequer, which I received accordingly some says afterwards. The same day I went to Mr. Banks, not finding him at home, I wrote a note to him desiring him to appoint me a day when I could have the favour to speak to him. For being appointed by His Majesty to go on the Expedition, I wished to be informed by him of particulars relative to my equipment. Having always been favoured with his friendship, I hoped he would add this to the many obligations he had already laid me under. It was just going away, and Dr. Scharter came in. I told of my going and began likewise his advice. He congratulated me coolly and was civil; but the moment I spoke of the matter, his coolness wore off, and all gloom was at once carried off upon their tases, or the really gave me some good information relative to many particulars.

**4 June 1772**  
*Weather: Clear*  
Mr. Banks sent me word that I might see him the next day. I named to see W. Banks and begged of him the favour to give me some informations relative to my Equipment, adding that I knew very well the necessity of common articles, but I shift that by his experience he must have found out the conscience or inconvenience of many things, which could not be frozen at such a distance, or did not occur to one who had not been on the Voyage. Mr. Banks declared he would give me any information and I should ask him Questions and he would must me.

**13 June 1772**  
*Weather: Clear*  
I told that as I was unacquainted with the peculiar and unfamiliar particulars, I only wished to be by him instructed in regard to them. But Mr. Touk declined, and I repeated that I should ask him questions and he would answer. A great many more company coming in, I on purpose asked only a few trifling questions and then applied to him for to let me see the drawings of his plants. He replied this would be of little use to me and excused himself with want at time. Then he said, since I saw, he declined to do it. I would not insist upon it hereupon. Mr. 13am asked me when the Ship would sell. I told I did not know, but suspected it would happen on Saturday next. Then he desired me to come Saturday next. I met Capt. that day at Mr. Banks's and went with nine to the Admiralty where a board of longitude was to be held, in order to see there Mr. Hormby. Then went with Cape Cook home, to wave him to dinner. Went to the Treasury & Exchequer & got 179 5. E pei me: 95 of which were to defray the Expences.

In [9]:
# Statistics: normalised dates and weather mentions in the compiled log
full_text = OUTPUT_PATH.read_text(encoding='utf-8')

# Normalised dates: **DD Month YYYY**
date_pattern = (
    r'\*\*\d{1,2} '
    r'(?:January|February|March|April|May|June|July|August|'
    r'September|October|November|December) \d{4}\*\*'
)
dates_found   = re.findall(date_pattern, full_text)

# Normalised weather: *Weather: ...*
weather_found = re.findall(r'\*Weather: ([^*]+)\*', full_text)

print(f'Normalised date headings : {len(dates_found)}')
for d in dates_found[:10]:
    print(f'  {d}')
if len(dates_found) > 10:
    print(f'  … and {len(dates_found)-10} more')

print()
print(f'Normalised weather notes : {len(weather_found)}')
for w in weather_found[:10]:
    print(f'  Weather: {w.strip()}')
if len(weather_found) > 10:
    print(f'  … and {len(weather_found)-10} more')

Normalised date headings : 41
  **12 May 1772**
  **4 May 1772**
  **27 May 1772**
  **32 June 1772**
  **16 June 1772**
  **4 June 1772**
  **4 June 1772**
  **4 June 1772**
  **13 June 1772**
  **16 June 1772**
  … and 31 more

Normalised weather notes : 41
  Weather: clear
  Weather: clear
  Weather: clear
  Weather: clear
  Weather: Clear
  Weather: Clear
  Weather: Clear
  Weather: Clear
  Weather: Clear
  Weather: Clear
  … and 31 more


---\n## Output\n\nThe compiled travel log is saved to:\n- **`data/travel_log_full_test.md`** — Markdown, human-readable\n- **`data/travel_log_entries.jsonl`** — one JSON record per batch (resumable)\n\n**Entry format:**\n```\n**DD Month YYYY**\n*Weather: <normalised term>*\n<coherent travel log prose in English>\n```